# 03 · Topics and titles in the sample

The sample is `general/outputs/sample.csv` from `01_sample.ipynb`: JWST galaxy journal articles and preprints,
2023 on, that **use JWST data** by the sample rule (candidate query, `bibgroup:JWST` or explicit data use
in the text, article or preprint, galaxy paper; see `01_sample.ipynb`).

1. **Discovery** (section 2): title phrases ranked by the change in their share of each year's titles.
2. **Topics** (sections 3–4): the topics of `common/trending_shared.py`, matched locally in title + abstract;
   share of papers and of citations per year.
3. **Catchy titles** (section 5): how often titles open with a playful phrase, and whether such papers are cited
   more.
4. **Export** (section 6): one row per sample paper (citations of this pull, topic flags, title label) for
   `04_citation_features.ipynb`.

## 1 · Setup and sample

In [ ]:
# Setup: repository root on the import path, shared settings and ADS helpers.
import os, sys, re, json, time, html, hashlib
from collections import Counter, defaultdict
from datetime import date, datetime
sys.path.insert(0, os.path.abspath(".."))
from common.config import require_texts
require_texts()   # this notebook reads ADS abstracts or full-text passages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from common.config import FREEZE, CALC_FROM_YEAR, SORT, REFRESH, GEN_OUT, GEN_PLOTS, data
from common.ads import bigquery as _bigquery, is_paper

SAMPLE_CSV         = os.path.join(GEN_OUT, "sample.csv")                    # from 01_sample.ipynb
TRENDING_CSV       = os.path.join(GEN_OUT, "trending_phrases_sample.csv")
TOPICS_JSON        = data("topics_sample_pull.json")                       # cache: bigquery pull per topic
CATCHY_CSV         = data("catchy_title_labels.csv")                        # hand labels per title
SAMPLE_TABLE_CSV   = os.path.join(GEN_OUT, "sample_table.csv")              # section 7, for 04
PLOTS_DIR          = GEN_PLOTS

def bigquery(q, bibcodes, fl="bibcode"):
    """ADS search restricted to the sample's bibcodes (sorted as the other searches)."""
    return _bigquery(q, bibcodes, fl=fl, sort=SORT)

def as_year(v):
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return None

# The sample: every paper of 01_sample.ipynb from CALC_FROM_YEAR on (all are articles or preprints).
sample = pd.read_csv(SAMPLE_CSV)
sample = sample[sample["year"] >= CALC_FROM_YEAR]
strict_rows = sample.to_dict("records")
strict_bibs = [r["bibcode"] for r in strict_rows]
print(f"Sample: {len(strict_rows):,} papers")

---
## 2 · Discovery: title n-grams, ranked by change in year-share

Every two- and three-word phrase of the sample's titles (phrases with a stop word or a word of one or two letters are
skipped) that appears in at least `MIN_TOTAL_MENTIONS` titles, ranked by the change in its share of titles from the
first two years to the last two, so that the growth of the field alone does not make a phrase rise. The topics of
section 3 were picked from this list.

In [ ]:
# Title phrases: stop words first.
TREND_STOP = set("""
    a an the of and or in on at to for with from by is are was were be been being this
    that these those as it its into over under between among not no but if than then so
    such we our us you your i they their he she his her which who whom what when where
    why how jwst nircam nirspec niriss miri jades ceers uncover primer cosmos-web ngdeep
    fresco glass data observations imaging spectroscopy spectra photometry galaxies galaxy
    survey using view evidence toward new first study analysis results sample
""".split())

# (year, title) for every sample paper up to the current year.
trend_rows = [(as_year(r["year"]), html.unescape(r["title"])) for r in strict_rows]
trend_rows = [(y, t) for y, t in trend_rows if y and CALC_FROM_YEAR <= y <= date.today().year]
n_by_year  = Counter(y for y, _ in trend_rows)
print(f"{len(trend_rows)} titles across {len(n_by_year)} years: {dict(sorted(n_by_year.items()))}")

# Phrases of n consecutive words, skipping any with a stop word or a word of 1-2 letters.
def trend_ngrams(title, n):
    words = re.findall(r"[a-zA-Z][a-zA-Z0-9+\-]*", title.lower())
    out = []
    for i in range(len(words) - n + 1):
        gram = words[i:i + n]
        if any(w in TREND_STOP or len(w) <= 2 for w in gram):
            continue
        out.append(" ".join(gram))
    return out

# Titles containing each 2- and 3-word phrase, per year.
trend_counts = defaultdict(Counter)
for y, title in trend_rows:
    seen = set()
    for n in (2, 3):
        for g in trend_ngrams(title, n):
            if g not in seen:
                seen.add(g)
                trend_counts[g][y] += 1

# Rank phrases with at least MIN_TOTAL_MENTIONS titles by the change in their share of titles, first
# two years against last two.
MIN_TOTAL_MENTIONS = 6
years_sorted = sorted(n_by_year)
early_years, recent_years = years_sorted[:2], years_sorted[-2:]

def trend_score(c):
    '''(change in year-share, recent share, early share) -- share of early_years' papers
    vs. share of recent_years' papers, so growth isn't just the corpus getting bigger.'''
    early  = sum(c.get(y, 0) for y in early_years)  / sum(n_by_year[y] for y in early_years)
    recent = sum(c.get(y, 0) for y in recent_years) / sum(n_by_year[y] for y in recent_years)
    return recent - early, recent, early

trend_rows_out = []
for g, c in trend_counts.items():
    if sum(c.values()) < MIN_TOTAL_MENTIONS:
        continue
    delta, recent, early = trend_score(c)
    row = {"phrase": g, "total": sum(c.values()), "early_share_pct": early * 100,
           "recent_share_pct": recent * 100, "delta_pp": delta * 100}
    row.update({str(y): c.get(y, 0) for y in years_sorted})
    trend_rows_out.append(row)

df_trending = pd.DataFrame(trend_rows_out).sort_values("delta_pp", ascending=False).reset_index(drop=True)
df_trending.to_csv(TRENDING_CSV, index=False)
print(f"{len(df_trending)} candidate phrases (>= {MIN_TOTAL_MENTIONS} mentions) -> {TRENDING_CSV}")
df_trending.head(25)

---
## 3 · The topic list, in the sample

The topic list of `common/trending_shared.py` (shared with the citation model), matched
locally in the title and abstract of each sample paper. A paper counts in
every topic it mentions, except that Black holes & AGN leaves out the Little Red Dot papers. The denominator for the
shares is the sample, per year. Citations come from one ADS bigquery of the sample on the freeze date (the citation
snapshot of the citation model too); a smaller sample later is restricted to those papers, never pulled again.

In [ ]:
# The topic list (common/trending_shared.py), matched locally on title and abstract.
from common.trending_shared import TOPICS, attention_table, attention_figures, topic_texts, topic_lists
years_axis = list(range(CALC_FROM_YEAR, date.today().year + 1))

# Citations of every sample paper: the bigquery pull of the freeze date (the citation snapshot of steps 4-9).
# The pull is restricted to the sample and never repeated after the freeze date: a sample paper it does not hold
# (one that passes on the texts ADS serves today) gets no citation count and is left out of the models.
cached = None
if not REFRESH and os.path.exists(TOPICS_JSON):
    with open(TOPICS_JSON, encoding="utf-8") as f:
        cached = json.load(f)
pulled = {d["bibcode"] for d in cached["corpus_docs"]} if cached and "corpus_docs" in cached else set()
after_freeze = bool(pulled) and date.today().isoformat() > FREEZE
if set(strict_bibs) <= pulled or after_freeze:
    keep = set(strict_bibs) & pulled
    corpus_docs = [d for d in cached["corpus_docs"] if d["bibcode"] in keep]
    print(f"Citations: the pull of {cached['saved'][:10]}, restricted to the {len(keep):,} sample papers "
          f"({len(pulled) - len(keep):,} pulled papers not in the sample now).")
    if len(keep) < len(strict_bibs):
        print(f"  {len(strict_bibs) - len(keep):,} sample papers are not in that pull: no citation count "
              f"(none is pulled after the freeze date).")
else:
    corpus_docs = bigquery("*:*", strict_bibs, fl="bibcode,year,citation_count")
    with open(TOPICS_JSON, "w", encoding="utf-8") as f:
        json.dump({"signature": hashlib.sha1("\n".join(sorted(strict_bibs)).encode()).hexdigest(),
                   "saved": datetime.now().isoformat(timespec="seconds"), "corpus_docs": corpus_docs},
                  f, ensure_ascii=False)
n_found = len(corpus_docs)
corpus_by_year, corpus_cites_by_year = Counter(), Counter()
for d in corpus_docs:
    corpus_by_year[as_year(d.get("year"))] += 1
    corpus_cites_by_year[as_year(d.get("year"))] += d.get("citation_count", 0) or 0
corpus_by_year = {y: corpus_by_year[y] for y in years_axis}
corpus_cites_by_year = {y: corpus_cites_by_year[y] for y in years_axis}

texts = topic_texts(strict_bibs)                  # title, abstract, publication date (fixed fields)
topic_docs = topic_lists(corpus_docs, texts)      # {topic: [docs]}

print(f"{n_found} of {len(strict_bibs)} sample bibcodes found in ADS "
      f"({len(strict_bibs) - n_found} not found, most likely re-bibcoded since 01_sample.ipynb ran)")
print("Sample papers per year (denominator):", corpus_by_year)
print("Citations to them, by publication year (denominator):", corpus_cites_by_year)
for topic, docs in topic_docs.items():
    print(f"  {topic}: {len(docs)} papers")

---
## 4 · All topics together: share of papers and share of citations, per year

Two figures with the same two panels (heatmap, and the same numbers as lines):

- **Papers:** % of that year's papers that mention the topic.
- **Citations:** % of all citations to that year's papers that go to papers on the topic. Comparing within a publication year removes the age effect, so years can be compared; the current year's papers have few citations yet, so that column is noisier.

A topic whose citation share is above its paper share draws more than its share of attention. The topic list is set in `common/trending_shared.py` and shared with the citation model: Black holes & AGN leaves out the Little Red Dot papers, so the two curves are disjoint (89% of LRD papers also mention black holes or AGN). Other topics overlap, so the shares don't add up to 100%. In the line chart both panels share one y axis, and the current, partial year is drawn dashed with an open marker.

Saved to `general/plots/topic_attention_sample.png` and `..._lines.png`, with the numbers in `topic_attention_sample.csv`.

In [ ]:
def by_year_counts(docs):
    papers, cites = Counter(), Counter()
    for d in docs:
        try:
            y = int(d.get("year"))
        except (TypeError, ValueError):
            continue
        papers[y] += 1
        cites[y] += d.get("citation_count", 0) or 0
    return papers, cites

# per plotted topic and year: share of papers and share of citations (trending_shared.attention_table)
flat_docs = topic_docs
df_topics_summary, topic_bibs = attention_table(flat_docs, corpus_by_year, corpus_cites_by_year, years_axis)
display(df_topics_summary.round(1))

# papers per topic
df_topics_all = pd.DataFrame([{"topic": t, "papers": len(d)} for t, d in flat_docs.items()])
display(df_topics_all)

os.makedirs(PLOTS_DIR, exist_ok=True)
attention_figures(df_topics_summary, topic_bibs, years_axis, date.today(),
                  sample_note=f"Sample: {len(strict_rows):,} JWST galaxy articles and preprints that use JWST data (in ADS's JWST "
                              f"bibliography or explicit data use in the text), matched in title + abstract.",
                  title="Which JWST galaxy topics are gaining attention",
                  out_prefix=f"{PLOTS_DIR}/topic_attention_sample")

---
## 5 · Catchy titles: a playful hook before the real title, or a playful title

Mostly titles that open with an intentionally funny phrase, wordplay or a cultural reference, followed by a separator (":", "—", "-", ".", "?") and the scientific title: *"Lord of LRDs: insights into a 'Little Red Dot'…"*, *"Here There Be (Dusty) Monsters: …"*, *"21 Balmer Jump Street: …"*. Titles that are playful as a whole, with no separator, count too: *"Five parameters are all you need (in ΛCDM)"*, *"Avoiding (photo-z) Catastrophe"*.

**How titles were labelled.** Every title in the sample with a separator (1,694) was split at the first separator. Rules set aside prefixes that are survey, programme, collaboration or software names (CEERS, JADES, "Early Results from GLASS-JWST", "Euclid preparation"), numbered series and descriptive science phrases. The remaining 411 prefixes, plus the filtered-out ones of 8 words or fewer, were then read and labelled (by Claude). The 1,292 titles without a separator were all read as well, and labelled as whole titles (column `form`: `prefix` or `whole title`):

- **catchy**: playful, figurative or cultural ("To be, or not to be", "Tip of the Iceberg", "BlackTHUNDER strikes twice");
- **borderline**: slogan-like, a nickname for an object, or a two-option question used as a hook ("The Cliff", "Cosmic Vine", "Unveiling the Cosmic Chemistry", "Quiescent or dusty?", "Too many or too massive?");
- **not**: descriptive science, other plain scientific questions, project / survey / series names, software.

Titles of the papers that entered the sample with the freeze (52) were labelled the same way (column `reviewed`) and reviewed by the author (one change); column `reviewed_by_user` marks every label the author reviewed.

**In the analysis a title is catchy or not: borderline titles count as not catchy.**

The labels are in `data/<freeze>/catchy_title_labels.csv` (hand-made; never regenerated by code). Papers added to the sample later are not labelled until reviewed; the cell below reports how many.

In [ ]:
# ── catchy titles: share per year ──
# Hand labels per title (catchy / borderline / not) joined to the sample papers; papers not yet
# labelled are counted and reported.
labels = pd.read_csv(CATCHY_CSV)[["bibcode", "label", "form"]].drop_duplicates("bibcode")

papers = pd.DataFrame([{"bibcode": r["bibcode"], "year": as_year(r["year"]), "title": r["title"]} for r in strict_rows])
# match a label through any identifier of the paper (a preprint labelled before publication keeps its
# label after ADS gives the published version a new bibcode)
lab_by_id = labels.set_index("bibcode")
def label_of(bibcode, identifiers):
    for i in [bibcode] + [x.strip() for x in str(identifiers or "").split("|") if x.strip()]:
        if i in lab_by_id.index:
            return lab_by_id.loc[i, "label"], lab_by_id.loc[i, "form"]
    return np.nan, np.nan
ids_of = dict(zip(sample["bibcode"], sample["identifiers"]))
papers[["label", "form"]] = [label_of(b, ids_of.get(b)) for b in papers["bibcode"]]
n_unlab = papers["label"].isna().sum()
papers = papers[papers["year"].isin(years_axis)]
papers["catchy"] = papers["label"].eq("catchy")
print(f"{len(papers)} papers; {papers['catchy'].sum()} catchy, "
      f"{(papers['label'] == 'borderline').sum()} borderline "
      f"({papers['catchy'].where(papers['form'].eq('whole title'), False).sum()} catchy as a whole title); "
      f"{n_unlab} not yet labelled (added since the review)")

# prevalence per year: share of catchy titles
prev = papers.groupby("year").agg(papers=("catchy", "size"), catchy=("catchy", "sum"))
prev["catchy %"] = 100 * prev["catchy"] / prev["papers"]
display(prev.round(1))



**Citations: catchy vs. other titles.** Year-matched bootstrap of the median: the catchy papers are resampled
within each publication year, and the same number of other papers is drawn from that year. Citations as of the ADS pull of this
notebook; the current year's papers have few citations yet and weigh little in the median difference.

In [ ]:
# ── citations: catchy vs. not, year-matched bootstrap ──
# Citations from the strict topic pull (the same snapshot as the topic shares); papers it did not
# return are dropped.
cites = {d["bibcode"]: d.get("citation_count", 0) or 0 for d in corpus_docs}
cp = papers.assign(citations=papers["bibcode"].map(cites)).dropna(subset=["citations"])
cp["group"] = np.where(cp["catchy"], "catchy", "not catchy")

def matched_boot(a, b, strata, n=5000, seed=0):
    """Median-citation difference a - b: a resampled within each stratum, the same number of b drawn
    from the same stratum. Returns (median difference, 2.5%, 97.5%, P(>0))."""
    rng = np.random.default_rng(seed)
    keys = sorted(set(map(tuple, a[strata].values)) & set(map(tuple, b[strata].values)))
    A = [a.loc[(a[strata] == k).all(axis=1), "citations"].to_numpy() for k in keys]
    B = [b.loc[(b[strata] == k).all(axis=1), "citations"].to_numpy() for k in keys]
    d = np.array([np.median(np.concatenate([x[rng.integers(0, len(x), len(x))] for x in A]))
                  - np.median(np.concatenate([y[rng.integers(0, len(y), len(x))] for x, y in zip(A, B)]))
                  for _ in range(n)])
    return np.median(d), *np.percentile(d, [2.5, 97.5]), (d > 0).mean()

# The two groups compared: catchy titles against all other titles (borderline counts as not catchy).
cat, oth = cp[cp["catchy"]], cp[~cp["catchy"]]

runs = [("catchy vs. not (by year)", cat, oth, ["year"])]
rows = []
for name, a, b, strata in runs:
    m, lo, hi, p = matched_boot(a, b, strata)
    rows.append({"comparison": name, "n catchy": len(a), "n other": len(b),
                 "median catchy": a["citations"].median(), "median other": b["citations"].median(),
                 "Δ median citations": m, "95% low": lo, "95% high": hi, "P(Δ>0)": p})
df_catchy_cites = pd.DataFrame(rows)
display(df_catchy_cites.round(2))
display(cp.groupby(["year", "group"])["citations"].agg(["size", "median"]).unstack())

## 6 · Export for the citation analysis

In [ ]:
# One row per sample paper: citations of this notebook's pull (the citation snapshot used from here on),
# whether ADS returned the paper in this pull, a flag per topic of the topic list, and the title label.
cites_now = {d["bibcode"]: d.get("citation_count", 0) or 0 for d in corpus_docs}
tab = sample[["bibcode", "title", "year", "doctype", "n_authors", "jwst_score", "jwst_bibgroup", "evidence"]].copy()
tab["in_topic_pull"] = tab["bibcode"].isin(cites_now)
tab["citations"] = tab["bibcode"].map(cites_now)
for t, docs in topic_docs.items():
    tab["topic: " + t] = tab["bibcode"].isin({d["bibcode"] for d in docs})
tab["catchy_label"] = tab["bibcode"].map(papers.set_index("bibcode")["label"])
tab["catchy_form"] = tab["bibcode"].map(papers.set_index("bibcode")["form"])
tab.to_csv(SAMPLE_TABLE_CSV, index=False)
print(f"-> {SAMPLE_TABLE_CSV}: {len(tab):,} papers, {tab['in_topic_pull'].sum():,} returned by the topic pull, "
      f"{tab['catchy_label'].notna().sum():,} with a title label")